# Model Signal Explorer
Use this notebook to check why your model is only sending BUY signals and to see the raw probabilities of each action.

In [22]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import MetaTrader5 as mt5

# Ensure we are in the right directory to import local modules
if 'backend/bot' not in os.getcwd():
    os.chdir('d:/Javascript/AutoTrader/backend/bot')

from model_config import get_config
from mt5_connector import get_ohlc_data, shutdown_mt5
from feature_eng import run_feature_pipeline
from ppo_agent import load_model, predict_action

print("Environment initialized.")

Environment initialized.


## 1. Load Model and Config
We will load the `v5` model specifically.

In [23]:
SYMBOL = "EURUSD"
VERSION = "v5"

cfg = get_config(SYMBOL, VERSION)
if not cfg:
    print(f"Config for {SYMBOL} {VERSION} not found!")
else:
    print(f"Loaded config: {cfg.name}")
    print(f"SL Options: {cfg.sl_options}")
    print(f"TP Options: {cfg.tp_options}")

model = load_model(cfg.model_path)
action_map = cfg.build_action_map()

Loaded config: PPO RL Agent v5
SL Options: [30, 50, 80]
TP Options: [60, 100, 160]
Successfully loaded PPO model from: models/model_eurusd_best_5.zip


## 2. Fetch Data and Prepare Features
We need to fetch enough data (300 bars) for feature engineering and extract a 120-bar window for the model.

In [18]:
# Fetch fresh data
ohlc_df = get_ohlc_data(cfg.symbol, cfg.timeframe, 300)
if ohlc_df is None:
    print("MT5 failed to return data. Make sure MT5 is open and connected.")
else:
    # Mock LLM features (since we are testing prediction logic, we use representative values)
    ohlc_df["bias_score"] = 0.5  # Neutral
    ohlc_df["confidence"] = 0.7
    ohlc_df["volatility"] = 0.5
    ohlc_df["trend_strength"] = 0.4
    ohlc_df["momentum"] = 0.2
    ohlc_df["skip_flag"] = 0.0

    # Run feature engineering
    featured_df, feature_cols = run_feature_pipeline(ohlc_df)

    # Add empty state features for local testing (assuming no open position)
    window_df = featured_df.iloc[-cfg.observation_window:].copy()
    window_df["position_state"] = 0.0
    window_df["time_in_trade_state"] = 0.0
    window_df["unrealized_pnl_state"] = 0.0
    
    final_features = window_df[feature_cols + ["position_state", "time_in_trade_state", "unrealized_pnl_state"]]
    print(f"Features ready. SHAPE: {final_features.shape}")

Features ready. SHAPE: (120, 17)


## 3. Predict and Analyze Probabilities
This section will show YOU what the model is thinking. If SELL is 0.0001% and BUY is 99%, you know it's biased.

In [19]:
action, action_probs = predict_action(model, final_features, window_size=cfg.observation_window)

# Create a summary table of probabilities
results = []
for i, prob in action_probs.items():
    act_tuple = action_map[i]
    act_type = act_tuple[0]
    direction = "N/A"
    sl = "N/A"
    tp = "N/A"
    
    if act_type == "OPEN":
        direction = "BUY" if act_tuple[1] == 1 else "SELL"
        sl = act_tuple[2]
        tp = act_tuple[3]
    
    results.append({
        "Action ID": i,
        "Type": act_type,
        "Direction": direction,
        "SL": sl,
        "TP": tp,
        "Probability": prob
    })

prob_df = pd.DataFrame(results).sort_values(by="Probability", ascending=False)

print("\n--- TOP 10 ACTIONS BY PROBABILITY ---")
print(prob_df.head(10).to_string(index=False))

print("\n--- PROBABILITY BY CATEGORY ---")
summary = prob_df.groupby(["Type", "Direction"])["Probability"].sum().reset_index()
print(summary.to_string(index=False))

# Check for SELL actions specifically
sell_prob = summary[summary["Direction"] == "SELL"]["Probability"].sum()
buy_prob = summary[summary["Direction"] == "BUY"]["Probability"].sum()
hold_prob = summary[summary["Type"] == "HOLD"]["Probability"].sum()

print(f"\nTOTAL SELL PROBABILITY: {sell_prob:.6f}")
print(f"TOTAL BUY PROBABILITY:  {buy_prob:.6f}")
print(f"TOTAL HOLD PROBABILITY: {hold_prob:.6f}")


--- TOP 10 ACTIONS BY PROBABILITY ---
 Action ID Type Direction    SL     TP  Probability
        18 OPEN       BUY  80.0  100.0     0.106900
        10 OPEN      SELL  80.0  160.0     0.083709
        14 OPEN       BUY  50.0   60.0     0.079116
        12 OPEN       BUY  30.0  100.0     0.075437
        11 OPEN       BUY  30.0   60.0     0.075115
         9 OPEN      SELL  80.0  100.0     0.059624
         3 OPEN      SELL  30.0  100.0     0.050105
        17 OPEN       BUY  80.0   60.0     0.043818
        15 OPEN       BUY  50.0  100.0     0.043288
         7 OPEN      SELL  50.0  160.0     0.042273

--- PROBABILITY BY CATEGORY ---
 Type Direction  Probability
CLOSE       N/A     0.036126
 HOLD       N/A     0.033632
 OPEN       BUY     0.523616
 OPEN      SELL     0.406626

TOTAL SELL PROBABILITY: 0.406626
TOTAL BUY PROBABILITY:  0.523616
TOTAL HOLD PROBABILITY: 0.033632


## 4. Manual Probability Override (Test if SELL works)
If you want to manually test if the backend accepts SELL signals, we can try to find the best SELL action index and see its parameters.

In [20]:
best_sell = prob_df[prob_df["Direction"] == "SELL"].iloc[0]
print(f"Best Sell Action: ID {best_sell['Action ID']}, Prob {best_sell['Probability']:.4f}")
print(f"Parameters: SL={best_sell['SL']}, TP={best_sell['TP']}")

# Shutdown MT5 when done
shutdown_mt5()

Best Sell Action: ID 10, Prob 0.0837
Parameters: SL=80.0, TP=160.0


### IMPORTANT: How to change TP
To change the TP to be smaller, you can edit `model_config.py` lines 72-73 for `v5`. 

**Before:**
```python
sl_options=[30, 50, 80],
tp_options=[60, 100, 160],
```

**After (Smaller TP):**
```python
sl_options=[30, 50, 80],
tp_options=[20, 40, 60],
```

The model will still output the same Action IDs, but when the bot maps them, it will use your new (smaller) TP values. Note that since the model wasn't trained with these exact values, the performance might vary, but it is the fastest way to reduce TP size without retraining.